# INITIAL IMPORT

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import numpy as np
import gymnasium as gym
from src.config import Configuration


CONFIG = Configuration(
    n_training_episodes = 1000,
    learning_rate = 0.7,
    n_eval_episodes = 1000,
    gym_id = "Taxi-v3",
    # gym_id = "MiniGrid-DoorKey-5x5-v0", 
    rm_file= "rm_taxi_v2.txt",
    # rm_file= "rm_doorkey.txt",
    max_steps = 1000,
    gamma = 0.99,
    max_epsilon = 1.0,
    min_epsilon = 0.05,
    decay_rate = 0.0001,
    use_rm = True, 
    use_crm = True, 
)

# LOAD ENV

In [3]:
import minigrid
from src.envs import MiniGridDiscreteWrapper

env = gym.make(CONFIG.gym_id, render_mode="rgb_array")
# env = MiniGridDiscreteWrapper(env)

state_space = env.observation_space.n
print("There are ", state_space, " possible states")
action_space = env.action_space.n
print("There are ", action_space, " possible actions")

There are  500  possible states
There are  6  possible actions


In [4]:
state, _ = env.reset()
# Returns a tuple: (taxi_row, taxi_col, passenger_location, destination)
taxi_row, taxi_col, passenger_location, destination = env.unwrapped.decode(state)

print(f"Taxi row: {taxi_row}")
print(f"Taxi column: {taxi_col}")
print(f"Passenger location: {passenger_location}")
print(f"Destination: {destination}")
print(f"State: {state}")

Taxi row: 1
Taxi column: 0
Passenger location: 0
Destination: 1
State: 101


# DEFINE Q-LEARNING

In [5]:
from src.models import RewardMachine
from src.envs import get_propositions_taxi, get_propositions_doorkey

get_propositions_fn = get_propositions_taxi if CONFIG.gym_id == "Taxi-v3" else get_propositions_doorkey

In [6]:
from src.models import QTable

In [7]:
from src.models import train_qtable

In [8]:
from src.models import evaluate_agent

# Generate and train

In [9]:
rm = RewardMachine(CONFIG, CONFIG.rm_file)
rm.states

{0: {(1, ('r',), 0.0), (2, ('g',), 0.0), (3, ('y',), 0.0), (4, ('b',), 0.0)},
 1: {(6, ('p', ' dg'), 5.0), (7, ('p', ' dy'), 5.0), (8, ('p', ' db'), 5.0)},
 2: {(5, ('p', ' dr'), 5.0), (7, ('p', ' dy'), 5.0), (8, ('p', ' db'), 5.0)},
 3: {(5, ('p', ' dr'), 5.0), (6, ('p', ' dg'), 5.0), (8, ('p', ' db'), 5.0)},
 4: {(5, ('p', ' dr'), 5.0), (6, ('p', ' dg'), 5.0), (7, ('p', ' dy'), 5.0)},
 5: {(5, ('!p',), -5.0), (9, ('d',), 10.0)},
 6: {(6, ('!p',), -5.0), (9, ('d',), 10.0)},
 7: {(7, ('!p',), -5.0), (9, ('d',), 10.0)},
 8: {(8, ('!p',), -5.0), (9, ('d',), 10.0)}}

In [10]:
events = get_propositions_fn(env, state)
# events = get_propositions_doorkey(env, state)

rm.step(events)

(1, 0.0, False)

In [11]:
def parse_state_taxi(state):
    taxi_row, taxi_col, passenger_location, destination = env.unwrapped.decode(state)
    # print((taxi_row, taxi_col))
    return (taxi_row, taxi_col)

In [12]:
qt = QTable(CONFIG, env, rm_file=CONFIG.rm_file if CONFIG.use_rm else None)
qt = train_qtable(
    CONFIG, 
    qt,  
    get_propositions_fn, 
    env,
    parse_state = parse_state_taxi,
    step_first_rm = True,  # Set to True to step RM immediately after reset
)

  0%|          | 0/1000 [00:00<?, ?it/s]

(2, 1) --1--> (1, 1) | 133, reward: -1
(1, 1) --2--> (1, 1) | 133, reward: -1
(1, 1) --0--> (2, 1) | 233, reward: -1
(2, 1) --2--> (2, 2) | 253, reward: -1
(2, 2) --1--> (1, 2) | 153, reward: -1
(1, 2) --1--> (0, 2) | 53, reward: -1
(0, 2) --0--> (1, 2) | 153, reward: -1
(1, 2) --4--> (1, 2) | 153, reward: -10
(1, 2) --0--> (2, 2) | 253, reward: -1
(2, 2) --4--> (2, 2) | 253, reward: -10
(2, 2) --4--> (2, 2) | 253, reward: -10
(2, 2) --3--> (2, 1) | 233, reward: -1
(2, 1) --5--> (2, 1) | 233, reward: -10
(2, 1) --2--> (2, 2) | 253, reward: -1
(2, 2) --3--> (2, 1) | 233, reward: -1
(2, 1) --1--> (1, 1) | 133, reward: -1
(1, 1) --4--> (1, 1) | 133, reward: -10
(1, 1) --0--> (2, 1) | 233, reward: -1
(2, 1) --1--> (1, 1) | 133, reward: -1
(1, 1) --0--> (2, 1) | 233, reward: -1
(2, 1) --3--> (2, 0) | 213, reward: -1
(2, 0) --2--> (2, 1) | 233, reward: -1
(2, 1) --5--> (2, 1) | 233, reward: -10
(2, 1) --1--> (1, 1) | 133, reward: -1
(1, 1) --4--> (1, 1) | 133, reward: -10
(1, 1) --2--> (1, 1

  5%|▌         | 52/1000 [00:00<00:05, 164.35it/s]

(0, 2) --4--> (0, 2) | 49, reward: -10
(0, 2) --5--> (0, 2) | 49, reward: -10
(0, 2) --0--> (1, 2) | 149, reward: -1
(1, 2) --5--> (1, 2) | 149, reward: -10
(1, 2) --3--> (1, 2) | 149, reward: -1
(1, 2) --3--> (1, 2) | 149, reward: -1
(1, 2) --1--> (0, 2) | 49, reward: -1
(0, 2) --0--> (1, 2) | 149, reward: -1
(1, 2) --3--> (1, 2) | 149, reward: -1
(1, 2) --5--> (1, 2) | 149, reward: -10
(1, 2) --2--> (1, 3) | 169, reward: -1
(1, 3) --2--> (1, 4) | 189, reward: -1
(1, 4) --4--> (1, 4) | 189, reward: -10
(1, 4) --1--> (0, 4) | 89, reward: -1
(0, 4) --4--> (0, 4) | 89, reward: -10
(0, 4) --5--> (0, 4) | 89, reward: -10
(0, 4) --4--> (0, 4) | 89, reward: -10
(0, 4) --5--> (0, 4) | 89, reward: -10
(0, 4) --1--> (0, 4) | 89, reward: -1
(0, 4) --4--> (0, 4) | 89, reward: -10
(0, 4) --1--> (0, 4) | 89, reward: -1
(0, 4) --3--> (0, 3) | 69, reward: -1
(0, 3) --0--> (1, 3) | 169, reward: -1
(1, 3) --4--> (1, 3) | 169, reward: -10
(1, 3) --1--> (0, 3) | 69, reward: -1
(0, 3) --2--> (0, 4) | 89, 

  7%|▋         | 69/1000 [00:00<00:05, 165.69it/s]

...

(3, 0) --1--> (2, 0) | 213, reward: -1
(2, 0) --2--> (2, 1) | 233, reward: -1
(2, 1) --2--> (2, 2) | 253, reward: -1
(2, 2) --3--> (2, 1) | 233, reward: -1
(2, 1) --0--> (3, 1) | 333, reward: -1
(3, 1) --1--> (2, 1) | 233, reward: -1
(2, 1) --3--> (2, 0) | 213, reward: -1
(2, 0) --2--> (2, 1) | 233, reward: -1
(2, 1) --4--> (2, 1) | 233, reward: -10
(2, 1) --2--> (2, 2) | 253, reward: -1
(2, 2) --2--> (2, 3) | 273, reward: -1
(2, 3) --1--> (1, 3) | 173, reward: -1
(1, 3) --5--> (1, 3) | 173, reward: -10
(1, 3) --5--> (1, 3) | 173, reward: -10
(1, 3) --1--> (0, 3) | 73, reward: -1
(0, 3) --3--> (0, 2) | 53, reward: -1
(0, 2) --2--> (0, 3) | 73, reward: -1
(0, 3) --5--> (0, 3) | 73, reward: -10
(0, 3) --1--> (0, 3) | 73, reward: -1
(0, 3) --1--> (0, 3) | 73, reward: -1
(0, 3) --3--> (0, 2) | 53, reward: -1
(0, 2) --2--> (0, 3) | 73, reward: -1
(0, 3) --3--> (0, 2) | 53, reward: -1
(0, 2) --3--> (0, 2) | 53, reward: -1
(0, 2) --2--> (0, 3) | 73, reward: -1
(0, 3) --5--> (0, 3) | 73, rewar

 98%|█████████▊| 980/1000 [00:05<00:00, 161.72it/s]

(0, 1) --4--> (0, 1) | 22, reward: -10
(0, 1) --1--> (0, 1) | 22, reward: -1
(0, 1) --2--> (0, 1) | 22, reward: -1
(0, 1) --3--> (0, 0) | 2, reward: -1
(0, 0) --4--> (0, 0) | 18, reward: -1
(0, 0) --5--> (0, 0) | 2, reward: -1
(0, 0) --0--> (1, 0) | 102, reward: -1
(1, 0) --1--> (0, 0) | 2, reward: -1
(0, 0) --3--> (0, 0) | 2, reward: -1
(0, 0) --0--> (1, 0) | 102, reward: -1
(1, 0) --0--> (2, 0) | 202, reward: -1
(2, 0) --5--> (2, 0) | 202, reward: -10
(2, 0) --5--> (2, 0) | 202, reward: -10
(2, 0) --0--> (3, 0) | 302, reward: -1
(3, 0) --2--> (3, 0) | 302, reward: -1
(3, 0) --1--> (2, 0) | 202, reward: -1
(2, 0) --4--> (2, 0) | 202, reward: -10
(2, 0) --2--> (2, 1) | 222, reward: -1
(2, 1) --0--> (3, 1) | 322, reward: -1
(3, 1) --1--> (2, 1) | 222, reward: -1
(2, 1) --1--> (1, 1) | 122, reward: -1
(1, 1) --3--> (1, 0) | 102, reward: -1
(1, 0) --2--> (1, 1) | 122, reward: -1
(1, 1) --3--> (1, 0) | 102, reward: -1
(1, 0) --4--> (1, 0) | 102, reward: -10
(1, 0) --5--> (1, 0) | 102, rewa

100%|██████████| 1000/1000 [00:06<00:00, 165.48it/s]

(2, 1) --2--> (2, 2) | 243, reward: -1
(2, 2) --1--> (1, 2) | 143, reward: -1
(1, 2) --5--> (1, 2) | 143, reward: -10
(1, 2) --2--> (1, 3) | 163, reward: -1
(1, 3) --5--> (1, 3) | 163, reward: -10
(1, 3) --4--> (1, 3) | 163, reward: -10
(1, 3) --3--> (1, 2) | 143, reward: -1
(1, 2) --2--> (1, 3) | 163, reward: -1
(1, 3) --5--> (1, 3) | 163, reward: -10
(1, 3) --2--> (1, 4) | 183, reward: -1
(1, 4) --3--> (1, 3) | 163, reward: -1
(1, 3) --4--> (1, 3) | 163, reward: -10
(1, 3) --4--> (1, 3) | 163, reward: -10
(1, 3) --3--> (1, 2) | 143, reward: -1
(1, 2) --2--> (1, 3) | 163, reward: -1
(1, 3) --2--> (1, 4) | 183, reward: -1
(1, 4) --2--> (1, 4) | 183, reward: -1
(1, 4) --3--> (1, 3) | 163, reward: -1
(1, 3) --0--> (2, 3) | 263, reward: -1
(2, 3) --4--> (2, 3) | 263, reward: -10
(2, 3) --0--> (3, 3) | 363, reward: -1
(3, 3) --4--> (3, 3) | 363, reward: -10
(3, 3) --0--> (4, 3) | 463, reward: -1
(4, 3) --3--> (4, 3) | 463, reward: -1
(4, 3) --4--> (4, 3) | 463, reward: -10
(4, 3) --3--> (4

In [13]:
mean_reward, std_reward = evaluate_agent(CONFIG, qt, get_propositions_fn, env)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

 38%|███▊      | 383/1000 [00:00<00:00, 938.58it/s]

100%|██████████| 1000/1000 [00:01<00:00, 952.90it/s]

Mean_reward=-200.00 +/- 0.00


100%|██████████| 1000/1000 [00:00<00:00, 14700.87it/s]

Mean_reward=7.87 +/- 2.62

In [14]:
qt._storage._table

{(0,
  (2, 1)): array([-96.46230665, -96.46706963, -96.45360012, -96.45460574,
        -96.44222841, -96.46404016]),
 (4,
  (1, 1)): array([-97.47965014, -97.51489424, -97.49568053, -97.47490396,
        -97.49888753, -97.49468358]),
 (1,
  (2, 1)): array([-97.46110284, -97.45725839, -97.43826173, -97.44514599,
        -97.40955955, -97.43544634]),
 (1,
  (1, 1)): array([-97.43545913, -97.4855989 , -97.46090342, -97.45298989,
        -97.46104465, -97.46045698]),
 (2,
  (2, 1)): array([-97.47417866, -97.46807531, -97.45343703, -97.44828532,
        -97.42955253, -97.45424425]),
 (2,
  (1, 1)): array([-97.45520525, -97.49865831, -97.4779818 , -97.46206834,
        -97.47999697, -97.47944174]),
 (3,
  (2, 1)): array([-97.48550352, -97.48109582, -97.46306018, -97.46600234,
        -97.44430056, -97.46864454]),
 (3,
  (1, 1)): array([-97.46965362, -97.50757619, -97.4915702 , -97.47149315,
        -97.4939384 , -97.49071516]),
 (4,
  (2, 1)): array([-97.49531843, -97.49043681, -97.4795163 ,

# Record video

In [15]:
from src.utils import record_video

record_video(CONFIG, qt, env, get_propositions_fn, video_name=f"{CONFIG.rm_file}_qtable_video.gif")

# Bench mark

In [16]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from src.config import Configuration
from src.models import QTable, evaluate_agent


def train_qtable_with_progress(config: Configuration, eval_every: int = 500):
    """Train and periodically evaluate to track learning evolution."""
    env = gym.make(config.gym_id, render_mode="rgb_array")
    qtable = QTable(config, env, rm_file="rm_taxi.txt" if config.use_rm else None)

    iterations = []
    eval_means = []
    eval_stds = []

    for episode in tqdm(range(config.n_training_episodes), desc=f"RM={config.use_rm}, CRM={config.use_crm}"):
        epsilon = config.min_epsilon + (config.max_epsilon - config.min_epsilon) * np.exp(-config.decay_rate * episode)
        state, _ = env.reset()
        qtable.reset_rm()

        terminated, truncated = False, False
        for _ in range(config.max_steps):
            action = qtable.epsilon_greedy_policy(state, epsilon, env)
            new_state, env_reward, terminated, truncated, _ = env.step(action)

            rm_done = qtable.update(
                state,
                action,
                env_reward,
                new_state,
                config.gamma,
                config.learning_rate,
                env,
                use_crm=config.use_crm,
                get_propositions=get_propositions_fn
            )

            if terminated or truncated or rm_done:
                break

            state = new_state

        if (episode + 1) % eval_every == 0 or episode == 0:
            mean_reward, std_reward = evaluate_agent(config, qtable, get_propositions_fn, env)
            iterations.append(episode + 1)
            eval_means.append(mean_reward)
            eval_stds.append(std_reward)

    env.close()
    return qtable, np.array(iterations), np.array(eval_means), np.array(eval_stds)


def convergence_iteration(iterations, values, smooth_window=3, plateau_ratio=0.95, patience=3):
    """Estimate convergence as first iteration that reaches and maintains near-plateau performance."""
    if len(values) == 0:
        return None

    if len(values) < smooth_window:
        smoothed = values
        smooth_iters = iterations
    else:
        kernel = np.ones(smooth_window) / smooth_window
        smoothed = np.convolve(values, kernel, mode="valid")
        smooth_iters = iterations[smooth_window - 1:]

    best = np.max(smoothed)
    threshold = best - (1.0 - plateau_ratio) * max(1.0, abs(best))

    if len(smoothed) < patience:
        return int(smooth_iters[np.argmax(smoothed)])

    for i in range(len(smoothed) - patience + 1):
        if np.all(smoothed[i : i + patience] >= threshold):
            return int(smooth_iters[i])

    return int(smooth_iters[np.argmax(smoothed)])


# Shared benchmark settings
base_kwargs = dict(
    n_training_episodes=4_000,
    learning_rate=0.7,
    n_eval_episodes=200,
    gym_id="Taxi-v3",
    max_steps=1000,
    gamma=0.90,
    max_epsilon=1.0,
    min_epsilon=0.05,
    decay_rate=0.001,
)

variants = [
    {"label": "No RM / No CRM", "use_rm": False, "use_crm": False},
    {"label": "RM / No CRM", "use_rm": True, "use_crm": False},
    {"label": "RM / CRM", "use_rm": True, "use_crm": True},
]

results = {}

for variant in variants:
    cfg = Configuration(
        **base_kwargs,
        use_rm=variant["use_rm"],
        use_crm=variant["use_crm"],
    )

    _, iters, means, stds = train_qtable_with_progress(cfg, eval_every=500)
    conv_it = convergence_iteration(iters, means, smooth_window=3, plateau_ratio=0.95, patience=3)

    results[variant["label"]] = {
        "iterations": iters,
        "mean_rewards": means,
        "std_rewards": stds,
        "convergence_iteration": conv_it,
    }

# Print convergence summary
for label, data in results.items():
    print(f"{label:15s} -> estimated convergence at episode {data['convergence_iteration']}")

# Plot evolution
plt.figure(figsize=(12, 6))
for label, data in results.items():
    x = data["iterations"]
    y = data["mean_rewards"]
    s = data["std_rewards"]

    plt.plot(x, y, marker="o", linewidth=2, label=label)
    plt.fill_between(x, y - s, y + s, alpha=0.12)

plt.title("Q-Learning Benchmark: Reward Evolution vs Training Episodes")
plt.xlabel("Training episodes")
plt.ylabel("Evaluation mean reward")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

RM=False, CRM=False:   0%|          | 0/4000 [00:00<?, ?it/s]


TypeError: QTable.update() missing 1 required positional argument: 'env'

# Human playing

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
from IPython.display import clear_output

# 1. Initialize with rgb_array for visual frames
if 'env' not in globals():
    env = gym.make(CONFIG.gym_id, render_mode="rgb_array")
    state, _ = env.reset()

# --- HUMAN PLAY ---
# 0=South, 1=North, 2=East, 3=West, 4=Pickup, 5=Dropoff
my_action = 5
# ------------------

# Step the simulation
state, reward, terminated, truncated, _ = env.step(my_action)

# 2. Get the single frame
img = env.render()

# 3. Visualize
clear_output(wait=True)
plt.figure(figsize=(5,5))
plt.imshow(img)
plt.axis('off')
plt.show()

# Data output
taxi_row, taxi_col, p_loc, dest = env.unwrapped.decode(state)
print(f"Action: {my_action} | Reward: {reward}")
print(f"Row: {taxi_row}, Col: {taxi_col}, P_Loc: {p_loc}, Dest: {dest}")

if terminated or truncated:
    print("Goal reached or Resetting...")
    state, _ = env.reset()